In [223]:
import sys
sys.path.append('../')

In [224]:
# !{sys.executable} -m pip install findicator
# !{sys.executable} -m pip install matplotlib
# !{sys.executable} -m pip install gym

In [225]:
from ice_fx.infrastructure.instrument_collection import instrumentCollection as ic
from ice_fx.oanda_api.oanda_api import OandaApi
from ice_fx.infrastructure.collect_data import run_collection
from findicator import (atr, bollinger_bands, macd, impulse_system, ema_channel,
                        elder_ray, stochastic_oscillator, force_index, rsi)
# from findicator import stochastic_oscillator
from chronofusion.utils.errors import InvalidArgumentError, NonOverlappingData, NotAvailableData, WrongMode

# api = OandaApi()
# ic.create_file(api.get_account_instruments(), "./test_env_data")
# ic.load_instruments("./test_env_data")
# ic.print_instruments()
from collections import deque, namedtuple
import torch
import torch.nn as nn
import time
import random
from tqdm import tqdm
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.preprocessing import StandardScaler

import numpy as np
import pandas as pd
import gym
from gym import spaces
# import talib
from typing import Tuple, Dict, List
import warnings
warnings.filterwarnings('ignore')
import joblib



Deep thought: We have a lot to do with it.

In [226]:
import pandas as pd
from pathlib import Path

# Robust data dir resolution (works when running notebook from repo root or from `test_env/`)
_data_dir = Path("./test_env_data")
if not _data_dir.exists():
    _data_dir = Path("./test_env/test_env_data")

m5_data = pd.read_pickle(_data_dir / "EUR_USD_M5.pkl")
m30_data = pd.read_pickle(_data_dir / "EUR_USD_M30.pkl")



In [227]:
m30_data_filtered = m30_data[['time', 'volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c']].copy()
m5_data_filtered = m5_data[['time', 'volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c']].copy()

In [228]:
115000/300

383.3333333333333

## Window Generator

In [229]:
class TimeFrameWindowGen:
    def __init__(self, 
                 higher_tf_data:pd.DataFrame, 
                 lower_tf_data:pd.DataFrame, 
                 higher_window_size:int = 50, 
                 lower_window_size:int = 50,
                 htf_start_index=0,
                 test=False,
                 n_candles_to_test:int=None):
        
        if test and not n_candles_to_test:
            raise InvalidArgumentError('Specify the n_candles_to_test!')
        # self.timestep = 0
        self.htf_data = higher_tf_data
        self.ltf_data = lower_tf_data
        self.h_window_size = higher_window_size
        self.l_window_size = lower_window_size
        self.lower_last_candle_index_for_test=None
        self.n_candles_to_test = n_candles_to_test
        self.htf_start_index = htf_start_index
        self.test = test
        dfs = self.cut_overlap(htf_start_index)
        self.htf_data = dfs['htf_data']
        self.ltf_data = dfs['ltf_data']
        self.alignment(htf_start_index=htf_start_index)

    def alignment(self, htf_start_index: int=None, ltf_start_index=None):
        if htf_start_index is not None:
            self.higher_last_candle_index = self.h_window_size - 1 + htf_start_index
            if self.higher_last_candle_index >= self.htf_data.shape[0]:
                raise NotAvailableData(index=self.higher_last_candle_index, data_frame_scale='higher')
            
            # time_to_align = self.htf_data['time'].iloc[self.higher_last_candle_index]
            # aligned_index = self.ltf_data.index[self.ltf_data["time"] == time_to_align]
            aligned_index = self.index_match(time_df=self.htf_data, 
                                            index_df=self.ltf_data,
                                            index_to_align=self.higher_last_candle_index)
            
            self.lower_last_candle_index = aligned_index[0]
            
            if self.test:
                self.lower_last_candle_index_for_test = aligned_index[0] + self.n_candles_to_test

        elif ltf_start_index is not None:
            self.lower_last_candle_index =  ltf_start_index - 1 # + self.l_window_size 
            if self.lower_last_candle_index >= self.ltf_data.shape[0]:
                raise NotAvailableData(index=self.lower_last_candle_index, data_frame_scale='lower')
            
            # aligned_index = self.index_match(time_df=self.ltf_data, 
            #                                 index_df=self.htf_data,
            #                                 index_to_align=self.lower_last_candle_index)

            
            self.higher_last_candle_index = self.nearest_past_index(index_to_align=self.lower_last_candle_index)
            if self.test:
                self.lower_last_candle_index_for_test = self.lower_last_candle_index + self.n_candles_to_test
    def index_match(self,time_df, index_df, index_to_align):
        time_to_align = time_df['time'].iloc[index_to_align]
        # print('time_to_align', time_to_align)
        aligned_index = index_df.index[index_df["time"] == time_to_align]
        if aligned_index.empty:
            raise NonOverlappingData()
        return aligned_index
    
    def nearest_past_index(self, index_to_align):
        time_to_align = self.ltf_data['time'].iloc[index_to_align]
        # print('time to aling*****************', time_to_align)
        return self.htf_data.loc[self.htf_data['time'] <= time_to_align, 'time'].idxmax()

    def reset(self, idx_high=None, idx_low=None):
        if idx_high is not None:
            self.alignment(htf_start_index=idx_high)
        elif idx_low is not None:
            self.alignment(ltf_start_index=idx_low)
        else:
            self.alignment(self.htf_start_index)

    
    def generate(self):
        
        start = self.higher_last_candle_index - self.h_window_size + 1
        end = self.higher_last_candle_index + 1
        if end >= self.htf_data.shape[0]:
            print('lower_window_start:', self.lower_last_candle_index - self.l_window_size + 1)
            print('lower_window_end:', self.lower_last_candle_index + 1)
            raise NotAvailableData(index=end, data_frame_scale='higher')
        if start >= self.htf_data.shape[0]:
            raise NotAvailableData(index=start, data_frame_scale='higher')
        
        higher_fraction = self.htf_data[start:end]

        start = self.lower_last_candle_index - self.l_window_size + 1 
        end = self.lower_last_candle_index + 1

        if end >= self.ltf_data.shape[0]:
           raise NotAvailableData(index=end, data_frame_scale='lower')
        if start >= self.ltf_data.shape[0]:
           raise NotAvailableData(index=start, data_frame_scale='lower')
        
        if self.test and end > self.lower_last_candle_index_for_test:
            raise WrongMode()
        
        lower_fraction = self.ltf_data[start:end]
            
        # self.timestep += 1
        self.lower_last_candle_index += 1

        higher_time_delta = self.htf_data['time'].iloc[self.higher_last_candle_index] - \
                            self.htf_data['time'].iloc[self.higher_last_candle_index-1]
        
        if self.htf_data['time'].iloc[self.higher_last_candle_index] + higher_time_delta <=\
           self.ltf_data['time'].iloc[self.lower_last_candle_index]:
            
            self.higher_last_candle_index += 1
        

        return higher_fraction, lower_fraction
    
    def get_current_value(self, column_names=['mid_c'], idx=0):
        """ Return the current vlaue of a column_name if idx, but for previous one 
        for example idx=1"""
        curr_price_index  = self.lower_last_candle_index - 1 - idx
        list_of_values = self.ltf_data[column_names].iloc[curr_price_index]
        return list(list_of_values)
    

    
    def get_test_df(self):
        if self.test:
            end = self.lower_last_candle_index_for_test
            lower_fraction_for_test = self.ltf_data[end-self.n_candles_to_test:end]
            return lower_fraction_for_test
        else:
            raise WrongMode(true_mode='test', wrong_mode='train')
        
    def cut_overlap(self, idx=None):
        """ Cutting overlapping pieces of dataframes"""
        beginning_idx = idx if idx else 0
        self.alignment(htf_start_index=beginning_idx)

        ltf_start = self.lower_last_candle_index - self.l_window_size + 1
        htf_start = self.higher_last_candle_index - self.h_window_size + 1

        idx_to_test = self.htf_data.shape[0]-1

        aligned_index = None
        while idx_to_test >= 0:
            try:
                aligned_index = self.index_match(
                    time_df=self.htf_data,
                    index_df=self.ltf_data,
                    index_to_align=idx_to_test,
                )
                break
            except NonOverlappingData:
                idx_to_test -= 1

        if aligned_index is None:
            raise NonOverlappingData()

        return {
            'htf_data': self.htf_data[htf_start:idx_to_test].reset_index(drop=True),
            'ltf_data': self.ltf_data[ltf_start:aligned_index[0]].reset_index(drop=True),
        }
    
    def valid_idx(self, walk_forward_window, idx=None, time_ratio=6):
        dfs = self.cut_overlap(idx=idx)
        h_max = dfs['htf_data'].shape[0]
        l_max = dfs['ltf_data'].shape[0]
        valid_idx_range = {'htf_range':(0+self.h_window_size, h_max-1 - walk_forward_window//time_ratio -1),
                           'ltf_range':(0+self.l_window_size, l_max - walk_forward_window - 2)}
        return valid_idx_range

        

        

In [230]:
window_generator = TimeFrameWindowGen(m30_data_filtered, m5_data_filtered,
                                      higher_window_size=5, lower_window_size=5)
window_generator.alignment(0)
print('lower_last', window_generator.lower_last_candle_index)
print('higher_last', window_generator.higher_last_candle_index)

for i in range(20):

    if i == 10:
        print('\n\n\n')
        window_generator.alignment(ltf_start_index=5000)
        
    h, l = window_generator.generate()
    
    curr_price = window_generator.get_current_value()[0]
    print(h, '\n')
    print(l, '\n')
    print(curr_price, '\n *************************')
    

lower_last 4
higher_last 4
                       time  volume    mid_o    mid_h    mid_l    mid_c
0 2016-01-07 00:00:00+00:00     335  1.07764  1.07832  1.07759  1.07790
1 2016-01-07 00:30:00+00:00     207  1.07786  1.07810  1.07744  1.07778
2 2016-01-07 01:00:00+00:00    1323  1.07776  1.07990  1.07748  1.07966
3 2016-01-07 01:30:00+00:00    1844  1.07968  1.08100  1.07931  1.08029
4 2016-01-07 02:00:00+00:00     854  1.08026  1.08086  1.07996  1.08018 

                       time  volume    mid_o    mid_h    mid_l    mid_c
0 2016-01-07 01:40:00+00:00     334  1.08027  1.08099  1.08027  1.08079
1 2016-01-07 01:45:00+00:00     387  1.08076  1.08097  1.07937  1.07992
2 2016-01-07 01:50:00+00:00     369  1.07994  1.08100  1.07992  1.08100
3 2016-01-07 01:55:00+00:00     267  1.08098  1.08098  1.08029  1.08029
4 2016-01-07 02:00:00+00:00     210  1.08026  1.08068  1.08026  1.08027 

1.08027 
 *************************
                       time  volume    mid_o    mid_h    mid_l    mid

In [231]:
list(m5_data_filtered[['mid_c', 'volume']].iloc[0])

[1.07786, 74.0]

In [232]:
m5_data_filtered[['mid_c', 'volume']].iloc[0]

mid_c      1.07786
volume    74.00000
Name: 0, dtype: float64

In [233]:
window_generator = TimeFrameWindowGen(m30_data_filtered[:100], m5_data_filtered[:600],
                                      higher_window_size=5, lower_window_size=5)
dfs = window_generator.cut_overlap()

In [234]:
window_generator.valid_idx(50)

{'htf_range': (5, 88), 'ltf_range': (5, 516)}

In [235]:
# dfs['ltf_data']

In [236]:
# dfs['htf_data'].head(50)

In [237]:
window_generator = TimeFrameWindowGen(m30_data_filtered, m5_data_filtered,
                                      higher_window_size=5, lower_window_size=5,
                                      test=True,
                                      n_candles_to_test=15)

test_df = window_generator.get_test_df()
# test_df


In [238]:
window_generator.reset()
for i in range(10):

    h, l = window_generator.generate()
    
    curr_price = window_generator.get_current_value()[0]
    print(h, '\n')
    print(l, '\n')
    print(curr_price, '\n *************************')

                       time  volume    mid_o    mid_h    mid_l    mid_c
0 2016-01-07 00:00:00+00:00     335  1.07764  1.07832  1.07759  1.07790
1 2016-01-07 00:30:00+00:00     207  1.07786  1.07810  1.07744  1.07778
2 2016-01-07 01:00:00+00:00    1323  1.07776  1.07990  1.07748  1.07966
3 2016-01-07 01:30:00+00:00    1844  1.07968  1.08100  1.07931  1.08029
4 2016-01-07 02:00:00+00:00     854  1.08026  1.08086  1.07996  1.08018 

                       time  volume    mid_o    mid_h    mid_l    mid_c
0 2016-01-07 01:40:00+00:00     334  1.08027  1.08099  1.08027  1.08079
1 2016-01-07 01:45:00+00:00     387  1.08076  1.08097  1.07937  1.07992
2 2016-01-07 01:50:00+00:00     369  1.07994  1.08100  1.07992  1.08100
3 2016-01-07 01:55:00+00:00     267  1.08098  1.08098  1.08029  1.08029
4 2016-01-07 02:00:00+00:00     210  1.08026  1.08068  1.08026  1.08027 

1.08027 
 *************************
                       time  volume    mid_o    mid_h    mid_l    mid_c
0 2016-01-07 00:00:00+00

In [239]:
window_generator.lower_last_candle_index_for_test

np.int64(19)

## Env

In [240]:

EPS = 1e-8

class MultiTimeframeForexEnv(gym.Env):
    """
    Multi-Timeframe Forex Trading Environment for DQN
    Actions: 0=close, 1=buy, 2=sell, 3=hold
    Termination: 2% loss or 6% profit on trade
    Features: Prevents future time leakage, supports walk-forward training
    """
    
    def __init__(self, 
                 lower_tf_data: pd.DataFrame,
                 higher_tf_data: pd.DataFrame,
                #  timeframe_ratio: int = 5,  # e.g., 5min/1min = 5
                 initial_balance: float = 10000, 
                 lot_size: float = 0.1,
                 stop_loss_pct: float = 0.02, 
                 take_profit_pct: float = 0.06,
                 lookback_window: int = 20, 
                 transaction_cost: float = 0.0001,
                 walk_forward_window: int = 2000,  # Episode length (in LTF steps)
                 test: bool = False,
                 n_candles_to_test=None,
                 shaping_weight: float = 0.0,   # 0 => pure PnL/equity reward
                 flat_time_penalty: float = 0.0 # optional penalty per step when flat
                 #validation_window: int = 500
                 ):    # Validation window size
        
        super(MultiTimeframeForexEnv, self).__init__()
        
        self.test = test

        # Data preparation
        self.lower_tf_data = lower_tf_data.reset_index(drop=True).copy()
        self.higher_tf_data = higher_tf_data.reset_index(drop=True).copy()
        self._add_indicators()

        # Resolve a stable data directory for scalers/artifacts.
        from pathlib import Path
        self.data_dir = Path("./test_env_data")
        if not self.data_dir.exists():
            self.data_dir = Path("./test_env/test_env_data")

        self.lower_tf_data, self.ltf_scaler = self._scale_dataframe(
            self.lower_tf_data,
            exclude_cols=["time"],
            scaler_address=str(self.data_dir / "lower_scaler.pkl"),
        )

        self.higher_tf_data, self.htf_scaler = self._scale_dataframe(
            self.higher_tf_data,
            exclude_cols=["time"],
            scaler_address=str(self.data_dir / "higher_scaler.pkl"),
        )


        self.window_generator = TimeFrameWindowGen(self.higher_tf_data, 
                                                   self.lower_tf_data,
                                                   higher_window_size=lookback_window, 
                                                   lower_window_size=lookback_window,
                                                   htf_start_index=0,
                                                   test=test,
                                                   n_candles_to_test=n_candles_to_test)
        
        dfs = self.window_generator.cut_overlap()
        self.higher_tf_data = dfs['htf_data']
        self.lower_tf_data = dfs['ltf_data']
        # self.timeframe_ratio = timeframe_ratio
        
        # Trading parameters
        
        self.initial_balance = initial_balance
        self.lot_size = lot_size
        self.stop_loss_pct = stop_loss_pct
        self.take_profit_pct = take_profit_pct
        self.lookback_window = lookback_window
        self.transaction_cost = transaction_cost
        
        # Walk-forward parameters
        self.walk_forward_window = walk_forward_window
        # self.validation_window = validation_window
        self.training_start = 0
        self.training_end = walk_forward_window
        # Reward configuration
        self.reward_scale = float(self.initial_balance)  # normalize reward to returns
        self.shaping_weight = float(shaping_weight)
        self.flat_time_penalty = float(flat_time_penalty)

        # Legacy fields (kept to avoid breaking older cells; no longer used as termination rewards)
        self.normalize_value = 1#self.initial_balance / 10
        self.big_reward = 0.0
        self.big_penalty = 0.0
        
        # Position tracking
        self.current_position = 0.0
        self.entry_price = None
        self.current_step = 0
        self.current_balance = initial_balance
        self.trade_history = []
        self.steps_in_trade = 0
        # self.step_not_in_trade = 0
        
        # Action space: 0=close, 1=buy, 2=sell, 3=hold
        self.action_space = spaces.Discrete(4)
        
        # State space (combo function output)
        # self.observation_space = spaces.Box(
        #     low=-np.inf, high=np.inf, 
        #     shape=(10,), dtype=np.float32
        # )
        
        # Episode tracking
        self.max_steps = len(self.lower_tf_data) - self.lookback_window - 1
        
        # Indicators cache for preventing future leakage
        # self.lower_indicators_cache = {}
        # self.higher_indicators_cache = {}
        
    
    def _add_indicators(self) -> pd.DataFrame:  # OK
        
        # TODO Add return column
        # TODO Scale the data!!!
        close_price = self.lower_tf_data['mid_c'].copy()
        high_price = self.lower_tf_data['mid_h'].copy()
        low_price = self.lower_tf_data['mid_l'].copy()
        volume = self.lower_tf_data['volume']

        self.lower_tf_data['force_index'] = force_index(close=close_price, volume=volume)
        # print(elder_ray(high=high_price, low=low_price, close=close_price).shape)
        self.lower_tf_data['elder_ray_bull'], self.lower_tf_data['elder_ray_bear'] = elder_ray(high=high_price, low=low_price, close=close_price)
        self.lower_tf_data['stochastic_k'], self.lower_tf_data['stochastic_d'] = stochastic_oscillator(high=high_price, low=low_price, close=close_price)
        self.lower_tf_data['rsi'] = rsi(close=close_price)
        self.lower_tf_data['ris'] = self.lower_tf_data['rsi']  # backward-compat (typo)
        self.lower_tf_data['returns'] = self.lower_tf_data['mid_c'].pct_change()
        self.lower_tf_data['returns_rolling_std'] = self.lower_tf_data['returns'].rolling(window=20).std()
        self.lower_tf_data['log_return'] = np.log(self.lower_tf_data['mid_c'] / self.lower_tf_data['mid_c'].shift(1))
        self.lower_tf_data['momentum'] = self.lower_tf_data['mid_c'] - self.lower_tf_data['mid_c'].shift(10)
        short_vol = self.lower_tf_data['returns'].rolling(window=10).std() + EPS
        long_vol = self.lower_tf_data['returns'].rolling(window=50).std() + EPS
        self.lower_tf_data['vol_ratio'] = short_vol / (long_vol)

        
        close_price = self.higher_tf_data['mid_c'].copy()
        
        self.higher_tf_data['ema_26'] = ema_channel(close_price, ema_period=26, use_atr=False)
        self.higher_tf_data['ema_13'] = ema_channel(close_price, ema_period=13, use_atr=False)
        _, _, self.higher_tf_data['macd_hist'] =  macd(close=close_price)
        self.higher_tf_data['impulse_system'] = impulse_system(close=close_price)
        self.higher_tf_data['returns'] = self.higher_tf_data['mid_c'].pct_change()
        self.higher_tf_data['returns_rolling_std'] = self.higher_tf_data['returns'].rolling(window=20).std()
        self.higher_tf_data['log_return'] = np.log(self.higher_tf_data['mid_c'] / self.higher_tf_data['mid_c'].shift(1))
        self.higher_tf_data['momentum'] = self.higher_tf_data['mid_c'] - self.higher_tf_data['mid_c'].shift(10)
        short_vol = self.higher_tf_data['returns'].rolling(window=10).std()
        long_vol = self.higher_tf_data['returns'].rolling(window=50).std()
        self.higher_tf_data['vol_ratio'] = short_vol / (long_vol + 1e-8)

        self.lower_tf_data.dropna(inplace=True)
        self.higher_tf_data.dropna(inplace=True)
        self.lower_tf_data.reset_index(drop=True, inplace=True)
        self.higher_tf_data.reset_index(drop=True, inplace=True)

    def _scale_dataframe(self, df, scaler_address, exclude_cols=['time']):
        """Scale all columns except those in exclude_cols.

        Notes:
        - Training mode (`test=False`): fit scaler on provided df, then save.
        - Test mode (`test=True`): load existing scaler and ONLY transform.

        This prevents refitting on test data.
        """

        # Separate features to scale and those to exclude
        cols_to_scale = [c for c in df.columns if c not in exclude_cols]

        if not self.test:
            scaler = StandardScaler()
            scaled_values = scaler.fit_transform(df[cols_to_scale])
            joblib.dump(scaler, scaler_address)
        else:
            scaler = joblib.load(scaler_address)
            scaled_values = scaler.transform(df[cols_to_scale])

        # Rebuild DataFrame
        scaled_df = pd.DataFrame(scaled_values, columns=cols_to_scale, index=df.index)
        
        # Reattach excluded columns
        for col in exclude_cols:
            scaled_df[col] = df[col]
        
        # Reorder to original
        scaled_df = scaled_df[df.columns]

        
        return scaled_df, scaler

    # def _inverse_column_value(self, scaled_value, scaler, col_name):
    #     cols_to_scale = [c for c in self.lower_tf_data.columns if c not in ["time"]]
    #     idx = cols_to_scale.index(col_name)
    #     mean = scaler.mean_[idx]
    #     scale = scaler.scale_[idx]
    #     return scaled_value * scale + mean

    def _inverse_column_values(self, scaled_values, scaler, col_names):
        """
        Inverses scaling for multiple columns.

        Args:
            scaled_values (list or np.ndarray): list of scaled values, same length as col_names
            scaler (StandardScaler): fitted scaler used to scale the data
            col_names (list of str): list of column names to inverse

        Returns:
            list: list of inversed (unscaled) values
        """
        cols_to_scale = [c for c in self.lower_tf_data.columns if c not in ["time"]]

        inversed_values = []
        for val, col_name in zip(scaled_values, col_names):
            idx = cols_to_scale.index(col_name)
            mean = scaler.mean_[idx]
            scale = scaler.scale_[idx]
            inversed_values.append(val * scale + mean)
        
        return inversed_values

    def set_walk_forward_window(self, start: int):  # OK
        """Align the environment to a new walk-forward start.

        `start` is interpreted as a **lower timeframe (LTF) start index** (M5 in this notebook),
        i.e. the first candle of the walk-forward segment.

        Because observations require `lookback_window` history, we align the internal
        window generator to `start + lookback_window`.
        """

        ltf_start_index = int(start) + int(self.lookback_window)
        self.window_generator.alignment(ltf_start_index=ltf_start_index)
    
    def reset(self): # OK 
        """Reset environment to initial state"""
        # self.window_generator.reset()
        self.current_balance = self.initial_balance
        self.current_position = 0.0
        self.entry_price = None
        self.current_step = 0
        # Start from training window beginning + lookback
        # if self.is_training:
        #     self.current_step = max(self.training_start + self.lookback_window, self.lookback_window)
        # else:
        #     self.current_step = max(self.training_end + self.lookback_window, self.lookback_window)
        
        self.trade_history = []
        self.steps_in_trade = 0
        
        return self._get_state()
    
    def _get_state(self): # OK
        """Get current state using combo function"""
        
        higher_candles, lower_candles = self.window_generator.generate()
        # higher_candles.drop(columns=["time"], inplace=True)
        # lower_candles.drop(columns=["time"], inplace=True)

        #state = self.combo(higher_candles, lower_candles)
        
        # data = (higher_candles, lower_candles)
        # Add position information
        position_info = np.array([
            self.current_position / self.lot_size,  # -1, 0, or +1
            (self._calculate_unrealized_pnl() / self.current_balance) * self.normalize_value,  # normalized unrealized P&L
        ], dtype=np.float32)
        
        # Replace last 2 features with position info
        # state[-2:] = position_info

        state = {
            'ltf_data': lower_candles,
            'htf_data': higher_candles,
            'position_info': position_info,
        }
        
        return state
    
    def _unrealized_pnl_at_price(self, price: float) -> float:
        """Unrealized PnL for current position at a given price (in account currency)."""
        if self.current_position == 0.0 or self.entry_price is None:
            return 0.0

        if self.current_position > 0:  # Long
            return (price - self.entry_price) * abs(self.current_position) * 100000
        else:  # Short
            return (self.entry_price - price) * abs(self.current_position) * 100000

    def _calculate_unrealized_pnl(self): # OK
        """Calculate unrealized profit/loss at the current env price."""
        return self._unrealized_pnl_at_price(self.get_current_price())
    
    def _get_valid_actions(self): # OK
        """Get valid actions based on current position"""
        if self.current_position == 0.0:  # Flat
            return [0, 1, 2]  # close(no-op), buy, sell
        else:  # In position
            return [0, 3]  # close, hold
    
    def step(self, action):
        """Execute one step in the environment.

        Reward is primarily profit-aligned: change in equity between timesteps,
        normalized by `initial_balance`.
        """

        # Price at time t (before advancing the window)
        price_t = self.window_generator.get_current_value(column_names=['mid_c'], idx=0)[0]
        price_t = self._inverse_column_values(
            scaled_values=[price_t],
            scaler=self.ltf_scaler,
            col_names=["mid_c"],
        )[0]

        equity_t = self.current_balance + self._unrealized_pnl_at_price(price_t)

        # Execute action at price_t
        shaping_reward = self._execute_action(action, price_t)

        # Advance one step in time
        self.current_step += 1
        if self.current_position != 0:
            self.steps_in_trade += 1

        # Generate next observation (advances window generator)
        next_state = self._get_state()

        # Price at time t+1 (after advancing)
        price_tp1 = self.get_current_price()

        # Check SL/TP on the new price and close if needed
        done = self._check_termination(price_tp1)

        # Time limit termination (end of walk-forward episode)
        if self.current_step >= self.walk_forward_window:
            done = True
            if self.current_position != 0:
                self._close_position(price_tp1)

        equity_tp1 = self.current_balance + self._unrealized_pnl_at_price(price_tp1)

        pnl_reward = (equity_tp1 - equity_t) / (self.reward_scale + 1e-12)
        reward = float(pnl_reward) + float(self.shaping_weight) * float(shaping_reward)

        # Optional penalty to discourage staying flat (use carefully)
        if self.flat_time_penalty and self.current_position == 0.0:
            reward -= float(self.flat_time_penalty)

        pos_type = 'No Position'
        if self.current_position != 0:
            pos_type = 'Long' if self.current_position > 0 else 'Short'

        info = {
            'current_balance': self.current_balance,
            'position_type': pos_type,
            'current_position_size': self.current_position,
            'unrealized_pnl': self._unrealized_pnl_at_price(price_tp1),
            'steps_in_trade': self.steps_in_trade,
        }

        return next_state, reward, int(done), info
    
    def _values_for_reward(self, needed_values_names:list, price_prev:bool=True):

        
        values_list = self.window_generator.get_current_value(needed_values_names,idx=0)
        original_values = self._inverse_column_values(scaled_values=values_list, 
                                    scaler=self.ltf_scaler,
                                    col_names=needed_values_names)
        
        momentum, returns_rolling_std, vol_ratio = original_values
        price_prev = self.window_generator.get_current_value(column_names=['mid_c'], idx=1)[0]
        price_prev = self._inverse_column_values(scaled_values=[price_prev], 
                                                   scaler=self.ltf_scaler, 
                                                   col_names=["mid_c"])[0]
        
        return momentum, returns_rolling_std, vol_ratio, price_prev
        
    def _execute_action(self, action, current_price):  # OK
        """Execute the given action and return reward"""
        needed_values_names = ['momentum', 'returns_rolling_std', 'vol_ratio']
        momentum, returns_rolling_std, vol_ratio, price_prev = self._values_for_reward(needed_values_names)
        
        valid_actions = self._get_valid_actions()
        if action not in valid_actions:
            return -0.0001  # Invalid action penalty
        
        
        reward = 0.0
        
        if action == 0:  # Close position
            if self.current_position != 0:
                reward = self._close_position(current_price)
            
        elif action == 1:  # Buy
            if self.current_position == 0:
                reward = self._open_position(current_price, self.lot_size)
            
        elif action == 2:  # Sell
            if self.current_position == 0:
                reward = self._open_position(current_price, -self.lot_size)
                

        elif action == 3:  # Hold
            if self.current_position != 0:

                ########### previous reward policy
                # # Small time penalty + unrealized P&L signal
                # unrealized_pnl = self._calculate_unrealized_pnl()
                # reward = (unrealized_pnl / self.initial_balance) * self.normalize_value - 0.001 #time penalty
                # if reward >= 0:
                #     reward  += reward * 0.2 # incentive for staying in profitable trades
                # else:
                #     reward -= reward * 0.2
                #############
                step_in_trade = self.steps_in_trade
                position_sign = np.sign(self.current_position)

                reward = self.compute_reward_row(
                    price=current_price,
                    price_prev=price_prev,
                    price_entry=self.entry_price,
                    step_in_trade=step_in_trade,
                    returns_rolling_std=returns_rolling_std,
                    momentum=momentum,
                    vol_ratio = vol_ratio,
                    position_sign=position_sign,
                    action_cost=self.transaction_cost,
                    exit_score=0.0,  # no exit yet
                )

                # optional reward scaling
                reward *= self.normalize_value
            
        return reward
    
    def compute_reward_row(self,
        price, price_prev, price_entry, step_in_trade,
        returns_rolling_std, momentum, vol_ratio,
        action_cost=0.0,  # transaction costs are accounted via balance/equity by default
        position_sign=+1,    # +1 long, -1 short
        Tmax=50,
        weights = dict(wR=0.5, wA=0.2, wT=0.15, wV=0.1, wE=0.1, wC=0.0),
        exit_score=0.0
    ):
        # simple/relative returns
        r_t = (price - price_prev) / (price_prev + EPS)
        R_entry = (price - price_entry) / (price_entry + EPS)

        sigma = returns_rolling_std + EPS
        

        # standardized quantities
        R_std = R_entry / sigma
        r_std = r_t / sigma
        M_std = momentum / sigma

        # regime score V_t
        Vt = np.tanh(M_std) * np.tanh(vol_ratio - 1.0)  # in (-1,1)

        # exit bonus (bounded)
        exit_bonus = np.tanh(exit_score)

        # bounded transforms
        BR = np.tanh(R_std)
        BA = np.tanh(position_sign * r_std)   # rewards favorable instantaneous move

        time_penalty = (step_in_trade / Tmax)  # grows to 1

        wR, wA, wT, wV, wE, wC = (weights[k] for k in ("wR","wA","wT","wV","wE","wC"))

        reward = (
            wR * BR +
            wA * BA -
            wT * time_penalty +
            wV * Vt * np.tanh(M_std) +
            wE * exit_bonus -
            wC * action_cost
        )
        return float(reward)
    

    def _open_position(self, price, position_size):  # OK 
        """Open a new position"""
        self.current_position = position_size
        self.entry_price = price
        self.steps_in_trade = 0
        
        # Transaction cost
        transaction_cost = abs(position_size) * 100000 * self.transaction_cost
        # print('Transaction cost', transaction_cost)
        self.current_balance -= transaction_cost
        
        # Opening a position shouldn't have an arbitrary reward; costs are reflected in equity.
        return 0.0
    
    def _close_position(self, price): # OK
        """Close current position and return realized P&L"""
        if self.current_position == 0:
            return 0.0

        position_sign = np.sign(self.current_position)
        pnl = (price - self.entry_price) * position_sign * abs(self.current_position) * 100000
        self.current_balance += pnl
        exit_score = np.sign(pnl) * np.tanh(abs(pnl) / self.initial_balance)
        

        needed_values_names = ['momentum', 'returns_rolling_std', 'vol_ratio']
        momentum, returns_rolling_std, vol_ratio, price_prev = self._values_for_reward(needed_values_names)

        base_reward = self.compute_reward_row(
            price=price,
            price_prev=price_prev,
            price_entry=self.entry_price,
            step_in_trade=self.steps_in_trade,
            returns_rolling_std=returns_rolling_std,
            momentum=momentum,
            vol_ratio=vol_ratio,
            position_sign=position_sign,
            action_cost=self.transaction_cost,
            exit_score=exit_score,
        )

        trade_reward = base_reward * self.normalize_value

        # transaction_cost = abs(self.current_position) * 100000 * self.transaction_cost
        
        success = (pnl > 0)

        self.trade_history.append({
            'entry_price': self.entry_price,
            'exit_price': price,
            'position':'Long' if self.current_position>0 else 'Short',
            'position_size': self.current_position,
            'pnl': pnl,
            'steps_in_trade': self.steps_in_trade,
            'success':success,
            # 'entry_step': self.current_step - self.steps_in_trade,
            'exit_step': self.current_step
        })
        

        # Reset position
        self.current_position = 0.0
        self.entry_price = None
        self.steps_in_trade = 0
        

        return  trade_reward
    
    def _check_termination(self, price: float) -> bool:
        """Check SL/TP termination at a given price.

        If SL/TP is hit, closes the position at `price` and returns True.
        """
        if self.current_position == 0.0:
            return False

        unrealized_pnl = self._unrealized_pnl_at_price(price)

        if unrealized_pnl <= -self.initial_balance * self.stop_loss_pct:
            self._close_position(price)
            return True

        if unrealized_pnl >= self.initial_balance * self.take_profit_pct:
            self._close_position(price)
            return True

        return False

    def action_to_name(self,act_num:int):
        act_dict = {0:'Close/No Op',
                    1:"Buy",
                    2:"Sell",
                    3:"Hold"}
        if act_num not in act_dict.keys():
            print(f"{act_num} is not a valid action!")
        else:
            return act_dict[act_num]
        
    def get_current_price(self):
        current_price = self.window_generator.get_current_value(column_names=['mid_c'], idx=0)[0]
        current_price = self._inverse_column_values(scaled_values=[current_price], 
                                                   scaler=self.ltf_scaler, 
                                                   col_names=["mid_c"])[0]
        return current_price
    
    def get_trade_statistics(self):
        """Get trading statistics"""
        if not self.trade_history:
            return {}
        
        trades_df = pd.DataFrame(self.trade_history)
        
        stats = {
            'total_trades': len(trades_df),
            'winning_trades': len(trades_df[trades_df['pnl'] > 0]),
            'losing_trades': len(trades_df[trades_df['pnl'] < 0]),
            'total_pnl': trades_df['pnl'].sum(),
            'average_pnl': trades_df['pnl'].mean(),
            'max_profit': trades_df['pnl'].max(),
            'max_loss': trades_df['pnl'].min(),
            'win_rate': len(trades_df[trades_df['pnl'] > 0]) / len(trades_df) * 100,
            'average_trade_duration': trades_df['steps_in_trade'].mean(),
            'sharpe_ratio': trades_df['pnl'].mean() / (trades_df['pnl'].std() + 1e-8) if len(trades_df) > 1 else 0.0,
            'final_balance': self.current_balance
        }
        
        return stats

# NOTE: example instantiation removed to keep this cell import-safe.
# Instantiate `MultiTimeframeForexEnv` in dedicated test/training cells instead.


In [241]:
import numpy as np
import pandas as pd

EPS = 1e-8

def compute_reward_row(
    price, price_prev, price_entry, step_in_trade,
    returns_rolling_std, momentum, short_vol, long_vol,
    action_cost=0.0005,  # e.g. 5 bps per step
    position_sign=+1,    # +1 long, -1 short
    Tmax=50,
    weights = dict(wR=0.5, wA=0.2, wT=0.15, wV=0.1, wE=0.1, wC=1.0),
    exit_score=0.0
):
    # simple/relative returns
    r_t = (price - price_prev) / (price_prev + EPS)
    R_entry = (price - price_entry) / (price_entry + EPS)

    sigma = returns_rolling_std + EPS
    std_short = short_vol + EPS
    std_long = long_vol + EPS

    # standardized quantities
    R_std = R_entry / sigma
    r_std = r_t / sigma
    M_std = momentum / sigma

    # regime score V_t
    vol_ratio = std_short / std_long
    Vt = np.tanh(M_std) * np.tanh(vol_ratio - 1.0)  # in (-1,1)

    # exit bonus (bounded)
    exit_bonus = np.tanh(exit_score)

    # bounded transforms
    BR = np.tanh(R_std)
    BA = np.tanh(position_sign * r_std)   # rewards favorable instantaneous move

    time_penalty = (step_in_trade / Tmax)  # grows to 1

    wR, wA, wT, wV, wE, wC = (weights[k] for k in ("wR","wA","wT","wV","wE","wC"))

    reward = (
        wR * BR +
        wA * BA -
        wT * time_penalty +
        wV * Vt * np.tanh(M_std) +
        wE * exit_bonus -
        wC * action_cost
    )
    return float(reward)

In [242]:
# NOTE (deprecated): this cell used to contain a duplicate top-level
# `_execute_action(self, ...)` implementation.
#
# The authoritative implementation lives inside `MultiTimeframeForexEnv`.
# Keeping a second, divergent copy here is a common source of confusion.
#
# (Intentionally left blank.)


In [243]:
# NOTE (deprecated): this cell used to contain a duplicate top-level
# `_close_position(self, ...)` implementation.
#
# The authoritative implementation lives inside `MultiTimeframeForexEnv`.
# (Intentionally left blank.)


## Walk Forward Generator

In [244]:
def create_walk_forward_trainer(
                               total_steps: int,
                               window_size: int = 2000,
                               overlap: int = 200):
    """
    Create walk-forward training windows
    
    Example:
    - Window 1: Train 0-2000, 
    - Window 2: Train 1800-3800, 
    - Window 3: Train 3600-5600
    """
    windows = []
    start = 0
    
    while start + window_size  < total_steps:
        train_start = start
        train_end = start + window_size
        
        windows.append({
            'train_start': train_start,
            'train_end': train_end,
        })
        
        start += window_size - overlap
    
    return windows


## Testing the Environment

In [245]:
lower_tf_data = m5_data_filtered
higher_tf_data = m30_data_filtered

env = MultiTimeframeForexEnv(
    lower_tf_data=lower_tf_data,
    higher_tf_data=higher_tf_data,
    walk_forward_window=100,
    stop_loss_pct=0.001,
    take_profit_pct=0.0011
)


windows = create_walk_forward_trainer(
    total_steps=8000,
    window_size=2000,
    overlap=200
)

# print(f"\n=== Walk-Forward Training Windows ===")
# for i, window in enumerate(windows):
#     print(f"Window {i+1}:")
#     print(f"  Train: {window['train_start']}-{window['train_end']}")

# Test first training window
# print(f"\n=== Testing Window 1 ===")

window = windows[0]
env.set_walk_forward_window(window['train_start'])

state = env.reset()
# print('start price:', env.get_current_price())
# print(f"State shape:\n{state['data'][1]['mid_c'].tail()}\n")

# Run training simulation
total_reward = 0
steps = 0
trades = 0

for step in range(1000):  # Test first 50 steps
    # Get valid actions and choose one
    valid_actions = env._get_valid_actions()
    # print('valid actions:', valid_actions)
    action = np.random.choice(valid_actions)
    # print('chosen action', env.action_to_name(action))
    next_state, reward, done, info = env.step(action)
    # print('\n\n')
    # print('current price:', env.get_current_price(), '\n')
    total_reward += reward
    steps += 1
    
    if len(env.trade_history) > trades:
        trades = len(env.trade_history)
        print(f"Step {steps}: Trade completed! Position={env.trade_history[-1]['position']}, Reward={reward:.6f}")
        print(f"Entry Price:{env.trade_history[-1]['entry_price']}, Exit Price:{env.trade_history[-1]['exit_price']}, Balance: ${info['current_balance']:.2f}")
        print(f"Price change: {env.trade_history[-1]['exit_price'] - env.trade_history[-1]['entry_price']}\n")
        # print(f"  Position: {info['current_position_size']}")
        # print(f"  Window Progress: {info['window_progress']:.1%}")
    
    if done:
        # print(next_state)
        print(f"Episode terminated at step {steps}")

        break

Step 2: Trade completed! Position=Short, Reward=0.000000
Entry Price:1.08682, Exit Price:1.0867, Balance: $10000.20
Price change: -0.00011999999999989797

Step 7: Trade completed! Position=Long, Reward=0.000000
Entry Price:1.0875, Exit Price:1.08704, Balance: $9994.60
Price change: -0.00045999999999990493

Step 13: Trade completed! Position=Short, Reward=0.000000
Entry Price:1.087, Exit Price:1.08727, Balance: $9990.90
Price change: 0.00026999999999999247

Step 17: Trade completed! Position=Long, Reward=0.000000
Entry Price:1.08692, Exit Price:1.08713, Balance: $9992.00
Price change: 0.00020999999999982144

Step 20: Trade completed! Position=Short, Reward=0.000000
Entry Price:1.08724, Exit Price:1.08671, Balance: $9996.30
Price change: -0.0005299999999999194

Step 22: Trade completed! Position=Short, Reward=0.000000
Entry Price:1.0865, Exit Price:1.0865, Balance: $9995.30
Price change: 0.0

Step 25: Trade completed! Position=Long, Reward=0.000000
Entry Price:1.08648, Exit Price:1.08674

In [246]:
0.00042999999999993044*10000

4.299999999999304

In [247]:
(1.08421 - 1.0869)*10000


-26.900000000000812

# Agent

In [248]:
# src/chronofusion/models/transformer.py


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return x


class SingleScaleEncoder(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        # Learnable CLS token (used as a summary embedding for the whole window)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))

        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dropout=dropout,
            batch_first=True  # ✅ Now everything is (batch, seq_len, embed_dim)
        )
        self.transformer_encoder = nn.TransformerEncoder(
            self.encoder_layer,
            num_layers=num_layers
        )

    def forward(self, x):
        """Encode a time window and append a CLS summary token.

        x: (batch_size, seq_len, input_dim)
        returns: (batch_size, seq_len+1, d_model)
        """
        x = self.input_proj(x)  # (B, S, d_model)

        # Append CLS token at the END so we can take out[:, -1, :] as the summary.
        cls = self.cls_token.expand(x.size(0), 1, -1)  # (B, 1, d_model)
        x = torch.cat([x, cls], dim=1)  # (B, S+1, d_model)

        # Add positional encoding (CLS gets its own final position)
        x = self.pos_encoder(x)

        seq_len = x.size(1)
        device = x.device

        # Causal mask: allow attending to past only.
        # With CLS at the end, CLS can attend to all prior timesteps,
        # while earlier timesteps cannot attend to CLS.
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=device) * float('-inf'),
            diagonal=1
        )

        out = self.transformer_encoder(x, mask=causal_mask)
        return out

class MultiScaleTransformer(nn.Module):
    def __init__(
        self,
        input_dim_low,
        input_dim_high,
        d_model,
        nhead,
        num_layers,
        output_dim=100
    ):
        super().__init__()
        self.low_encoder = SingleScaleEncoder(input_dim_low, d_model, nhead, num_layers)
        self.high_encoder = SingleScaleEncoder(input_dim_high, d_model, nhead, num_layers)

        # Fusion: concat both representations per timestep + projection to output_dim
        self.fusion_proj = nn.Linear(d_model * 2, output_dim)

    def forward(self, x_low, x_high):
        """
        x_low: (batch_size, seq_len, input_dim_low)  # e.g., 30min timeframe window
        x_high: (batch_size, seq_len, input_dim_high) # e.g., 5min timeframe window
        """
        enc_low = self.low_encoder(x_low)    # (batch_size, seq_len, d_model)
        enc_high = self.high_encoder(x_high) # (batch_size, seq_len, d_model)

        combined = torch.cat([enc_low, enc_high], dim=-1)  # (batch_size, seq_len, d_model*2)
        out = self.fusion_proj(combined)                   # (batch_size, seq_len, output_dim=100)
        # out = out.mean(dim=1)
        out = out[:, -1, :]
        return out


In [249]:
torch.cuda.is_available()

True

In [250]:
lower_tf_data = m5_data_filtered
higher_tf_data = m30_data_filtered

env = MultiTimeframeForexEnv(
    lower_tf_data=lower_tf_data,
    higher_tf_data=higher_tf_data,
    walk_forward_window=2000,
)


windows = create_walk_forward_trainer(
    total_steps=8000,
    window_size=2000,
    overlap=200
)


window = windows[0]
env.set_walk_forward_window(window['train_start'])

state = env.reset()

# total_reward = 0
# steps = 0
# trades = 0

# for step in range(50): 
#     valid_actions = env._get_valid_actions()
#     action = np.random.choice(valid_actions)
#     next_state, reward, done, info = env.step(action)
#     total_reward += reward
#     steps += 1

In [251]:
ltf_window = state['ltf_data']
htf_window = state['htf_data']

agent = MultiScaleTransformer(input_dim_high=htf_window.shape[1]-1, input_dim_low=ltf_window.shape[1]-1,
                              d_model=256, nhead=4, num_layers=5, output_dim=100)


ltf_numeric = ltf_window.select_dtypes(include=["number"])
htf_numeric = htf_window.select_dtypes(include=["number"])

x_low = torch.tensor(ltf_numeric.values, dtype=torch.float32)
x_high = torch.tensor(htf_numeric.values, dtype=torch.float32)

out = agent(x_low.unsqueeze(0), x_high.unsqueeze(0))
out.shape

torch.Size([1, 100])

## QModel

In [252]:
import torch
import torch.nn as nn

class QModel(nn.Module):
    def __init__(self, 
                 input_dim_low, input_dim_high, 
                 d_model=256, nhead=4, num_layers=5,
                 fusion_dim=100,
                 pos_info_dim=10):  # <- length of extra array
        super().__init__()

        # Multi-scale transformer backbone
        self.backbone = MultiScaleTransformer(
            input_dim_low=input_dim_low,
            input_dim_high=input_dim_high,
            d_model=d_model,
            nhead=nhead,
            num_layers=num_layers,
            output_dim=fusion_dim
        )

        # Regression head: (100 + extra_dim) → ... → 4
        self.reg_head = nn.Sequential(
            nn.Linear(fusion_dim + pos_info_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4)   # 4 outputs for regression
        )

    def forward(self, x_low, x_high, pos_info):
        """
        x_low: (batch, seq_len, input_dim_low)
        x_high: (batch, seq_len, input_dim_high)
        pos_info: (batch, extra_dim)
        """
        fused = self.backbone(x_low, x_high)  # (batch, 100)

        # concat with extra input
        combined = torch.cat([fused, pos_info], dim=-1)  # (batch, 100+extra_dim)

        # regression output
        out = self.reg_head(combined)  # (batch, 4)

        return out


In [253]:
# suppose extra feature vector has length 8


# batch_size = 2
# x_low = torch.randn(batch_size, 20, 10)   # ltf_window
# x_high = torch.randn(batch_size, 20, 11)  # htf_window
# extra = torch.randn(batch_size, 8)        # extra array

ltf_window = state['ltf_data']
htf_window = state['htf_data']

ltf_numeric = ltf_window.select_dtypes(include=["number"])
htf_numeric = htf_window.select_dtypes(include=["number"])

x_low = torch.tensor(ltf_numeric.values, dtype=torch.float32)
x_high = torch.tensor(htf_numeric.values, dtype=torch.float32)
extra = torch.tensor(state['position_info'], dtype=torch.float32)


model = QModel(
    input_dim_low=x_low.shape[1], 
    input_dim_high=x_high.shape[1],
    pos_info_dim=extra.shape[0]
)

out = model(x_low.unsqueeze(0), x_high.unsqueeze(0), extra.unsqueeze(0))
print(out.shape)  # torch.Size([2, 4])


torch.Size([1, 4])


In [254]:
out

tensor([[-0.0936,  0.0335, -0.1121,  0.0206]], grad_fn=<AddmmBackward0>)

# DQN

In [ ]:
MEMORY_SIZE = 30_000     # size of memory buffer
GAMMA = 0.995             # discount factor
ALPHA = 1e-6              # learning rate  
NUM_STEPS_FOR_UPDATE = 4  # perform a learning update every C time steps
LOOKBACK_WINDOW_SIZE = 26
WALK_FORWARD_WINDOW = 50

STOP_LOSS_PCT = 0.001
TAKE_PROFIT_PCT = 0.003

SEED = 0  # Seed for the pseudo-random number generator.
MINIBATCH_SIZE = 64  # Mini-batch size.
TAU = 1e-4  # Soft update parameter.
E_DECAY = 0.995  # ε-decay rate for the ε-greedy policy.
E_MIN = 0.01 # 0.01  # Minimum ε value for the ε-greedy policy.

In [256]:
# state_size = env.observation_space.shape
num_actions = env.action_space.n
# print(num_actions)

In [257]:
# m30_data_filtered[-10:-1]

In [258]:
# m5_data_filtered[-10:-1]

In [259]:
def pack_state(state):
    """Pack a dict-state into compact numeric arrays for replay memory.

    Returns:
        (x_low_np, x_high_np, pos_np) as float32 numpy arrays.
    """
    ltf_window = state["ltf_data"]
    htf_window = state["htf_data"]

    ltf_numeric = ltf_window.select_dtypes(include=["number"])
    htf_numeric = htf_window.select_dtypes(include=["number"])

    if "position_info" not in state:
        raise KeyError("state must contain 'position_info'")

    pos = state["position_info"]

    x_low_np = ltf_numeric.values.astype("float32", copy=False)
    x_high_np = htf_numeric.values.astype("float32", copy=False)
    pos_np = np.asarray(pos, dtype="float32")

    return x_low_np, x_high_np, pos_np


def state_to_tensor(state, device="cpu"):
    """Convert a state (dict OR packed tuple) to model input tensors."""

    if isinstance(state, dict):
        x_low_np, x_high_np, pos_np = pack_state(state)
    else:
        # packed (x_low_np, x_high_np, pos_np)
        x_low_np, x_high_np, pos_np = state

    x_low = torch.as_tensor(x_low_np, dtype=torch.float32, device=device)
    x_high = torch.as_tensor(x_high_np, dtype=torch.float32, device=device)
    pos_info = torch.as_tensor(pos_np, dtype=torch.float32, device=device)

    return x_low.unsqueeze(0), x_high.unsqueeze(0), pos_info.unsqueeze(0)


In [260]:
def states_list_to_tensors(states, device='cpu'):
    """
    Converts a batch of states (list of dicts) into batched tensors
    suitable for NN input.

    Each state dict has:
        - 'ltf_data': DataFrame
        - 'htf_data': DataFrame
        - 'position_info': list or array
    
    Returns:
        x_low:  Tensor [batch_size, L, num_features_low]
        x_high: Tensor [batch_size, H, num_features_high]
        pos_info: Tensor [batch_size, pos_dim]
    """
    x_low_list = []
    x_high_list = []
    pos_list = []

    for state in states:
        if isinstance(state, dict):
            x_low_np, x_high_np, pos_np = pack_state(state)
        else:
            x_low_np, x_high_np, pos_np = state

        x_low_list.append(torch.as_tensor(x_low_np, dtype=torch.float32, device=device))
        x_high_list.append(torch.as_tensor(x_high_np, dtype=torch.float32, device=device))
        pos_list.append(torch.as_tensor(pos_np, dtype=torch.float32, device=device))

    # Stack into batch dimension
    x_low = torch.stack(x_low_list, dim=0)    # [B, L, low_feats]
    x_high = torch.stack(x_high_list, dim=0)  # [B, H, high_feats]
    pos_info = torch.stack(pos_list, dim=0)   # [B, pos_dim]

    return x_low, x_high, pos_info


In [261]:
2990/50

59.8

In [262]:
# higher_tf_data.tail(50)
higher_tf_data.tail(25)

,time,volume,mid_o,mid_h,mid_l,mid_c
115859,2025-04-30 11:30:00+00:00,4151,1.13702,1.13808,1.13702,1.13733
115860,2025-04-30 12:00:00+00:00,5499,1.13734,1.13854,1.13700,1.13776
115861,2025-04-30 12:30:00+00:00,13322,1.13766,1.13888,1.13650,1.13735
115862,2025-04-30 13:00:00+00:00,6942,1.13736,1.13766,1.13613,1.13646
115863,2025-04-30 13:30:00+00:00,6837,1.13646,1.13778,1.13634,1.13691
115864,2025-04-30 14:00:00+00:00,8906,1.13692,1.13732,1.13497,1.13622
115865,2025-04-30 14:30:00+00:00,11879,1.13624,1.13794,1.13513,1.13672
115866,2025-04-30 15:00:00+00:00,11371,1.13671,1.13753,1.13282,1.13319
115867,2025-04-30 15:30:00+00:00,9641,1.13319,1.13490,1.13233,1.13482
115868,2025-04-30 16:00:00+00:00,4964,1.13480,1.13574,1.13450,1.13533


In [263]:
lower_tf_data.tail(15)

,time,volume,mid_o,mid_h,mid_l,mid_c
693635,2025-04-30 22:45:00+00:00,113,1.13258,1.13278,1.13254,1.13254
693636,2025-04-30 22:50:00+00:00,129,1.13253,1.13270,1.13250,1.13268
693637,2025-04-30 22:55:00+00:00,158,1.13266,1.13280,1.13256,1.13256
693638,2025-04-30 23:00:00+00:00,239,1.13257,1.13257,1.13224,1.13248
693639,2025-04-30 23:05:00+00:00,158,1.13248,1.13248,1.13220,1.13222
693640,2025-04-30 23:10:00+00:00,253,1.13224,1.13230,1.13198,1.13200
693641,2025-04-30 23:15:00+00:00,251,1.13200,1.13238,1.13190,1.13212
693642,2025-04-30 23:20:00+00:00,136,1.13212,1.13244,1.13203,1.13240
693643,2025-04-30 23:25:00+00:00,165,1.13238,1.13246,1.13222,1.13243
693644,2025-04-30 23:30:00+00:00,236,1.13244,1.13261,1.13222,1.13260


In [264]:
# lower_tf_data.shape

In [265]:
n_candle_to_train = 3000
lower_tf_data = m5_data_filtered[:n_candle_to_train].copy()
higher_tf_data = m30_data_filtered[:n_candle_to_train//6].copy()


env = MultiTimeframeForexEnv(
    lower_tf_data=lower_tf_data,
    higher_tf_data=higher_tf_data,
    walk_forward_window=WALK_FORWARD_WINDOW,
    lookback_window=LOOKBACK_WINDOW_SIZE,
    # walk_forward_window=100,
    stop_loss_pct=STOP_LOSS_PCT, #0.0015,
    take_profit_pct=TAKE_PROFIT_PCT
)


current_state = env.reset()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

x_low, x_high, pos_info = state_to_tensor(current_state, device=device)

In [266]:
lower_tf_data.describe()

,volume,mid_o,mid_h,mid_l,mid_c
count,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000
mean,132.878667,1.088356,1.088638,1.088081,1.088360
std,125.758397,0.003457,0.003471,0.003442,0.003452
min,1.000000,1.077310,1.077720,1.077130,1.077300
25%,54.000000,1.086200,1.086460,1.085915,1.086220
50%,101.000000,1.088545,1.088800,1.088300,1.088540
75%,168.000000,1.090480,1.090760,1.090140,1.090480
max,1964.000000,1.098110,1.098460,1.097520,1.098070


In [267]:
# print(x_low.device, x_high.device, pos_info.device)
env.lower_tf_data.columns

Index(['time', 'volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'force_index',
       'elder_ray_bull', 'elder_ray_bear', 'stochastic_k', 'stochastic_d',
       'rsi', 'ris', 'returns', 'returns_rolling_std', 'log_return',
       'momentum', 'vol_ratio'],
      dtype='object')

In [268]:
lower_tf_data.shape

(3000, 6)

In [269]:

q_network = QModel(
    input_dim_low=x_low.shape[2], 
    input_dim_high=x_high.shape[2],
    pos_info_dim=pos_info.shape[1],
    fusion_dim=64
).to(device)


target_q_network = QModel(
    input_dim_low=x_low.shape[2], 
    input_dim_high=x_high.shape[2],
    pos_info_dim=pos_info.shape[1],
    fusion_dim=64
).to(device)

optimizer = optim.Adam(q_network.parameters(), lr=ALPHA)



In [270]:
print(device)

cuda


In [271]:
q_vals = q_network(x_low, x_high, pos_info)
# np.argmax(q_vals.detach().numpy()[0])

In [272]:
experience = namedtuple("Experience", field_names=["state", "action", "reward", "next_state", "done"])

In [273]:
mse_loss = nn.SmoothL1Loss() #nn.MSELoss()

def compute_loss(experiences, gamma, q_network, target_q_network, device="cpu"):
    """ 
    Calculates the loss.
    
    Args:
      experiences: (tuple) tuple of ["state", "action", "reward", "next_state", "done"] namedtuples
      gamma: (float) The discount factor.
      q_network: (tf.keras.Sequential) Keras model for predicting the q_values
      target_q_network: (tf.keras.Sequential) Keras model for predicting the targets
          
    Returns:
      loss: (TensorFlow Tensor(shape=(0,), dtype=int32)) the Mean-Squared Error between
            the y targets and the Q(s,a) values.
    """
    
    states, actions, rewards, next_states, dones = experiences
    # states = states.to(device)
    x_low_current, x_high_current, pos_info_current = states_list_to_tensors(states, device=device)
    x_low_next, x_high_next, pos_info_next = states_list_to_tensors(next_states, device=device)
    actions = actions.to(device)
    rewards = rewards.to(device)
    # next_states = next_states.to(device)
    dones = dones.to(device)

    # Q(s', a') from target network
    with torch.no_grad():
        max_qsa = target_q_network(x_low_next, x_high_next, pos_info_next).max(dim=1)[0]
        y_targets = rewards + gamma * max_qsa * (1 - dones)
    # print(y_targets)
    # Q(s,a) from current network
    q_values = q_network(x_low_current, x_high_current, pos_info_current)                          # shape: [batch, num_actions]
    q_values = q_values.gather(1, actions.unsqueeze(1))   # shape: [batch, 1]
    q_values = q_values.squeeze(1)                        # shape: [batch]
    # print(q_values)
    # Loss
    loss = mse_loss(q_values, y_targets)

    return loss

In [274]:
def update_target_network(q_network, target_q_network, tau=1.0):
    """
    Updates the weights of the target Q-Network using a soft update.
    
    The weights of the target_q_network are updated using the soft update rule:
    
                    w_target = (TAU * w) + (1 - TAU) * w_target
    
    where w_target are the weights of the target_q_network, TAU is the soft update
    parameter, and w are the weights of the q_network.
    
    Args:
        q_network (tf.keras.Sequential): 
            The Q-Network. 
        target_q_network (tf.keras.Sequential):
            The Target Q-Network.
            
    Copies weights from q_network to target_q_network.
    tau=1.0 -> hard update
    tau<1.0 -> soft update (Polyak averaging)
    """
    # print('networks updated')
    for target_param, local_param in zip(target_q_network.parameters(), q_network.parameters()):
        target_param.data.copy_(tau * local_param.data + (1.0 - tau) * target_param.data)

In [275]:
# @torch.func
def agent_learn(experiences, gamma, device="cpu"):
    """
    Updates the weights of the Q networks.
    
    Args:
      experiences: (tuple) tuple of ["state", "action", "reward", "next_state", "done"] namedtuples
      gamma: (float) The discount factor.
    
    """
    
    loss = compute_loss(experiences, gamma, q_network, target_q_network, device=device)
    # print(f' loss {loss}')
    optimizer.zero_grad()
    loss.backward()

    total_norm = torch.nn.utils.clip_grad_norm_(q_network.parameters(), max_norm=10.0)
    
    # Log when significant clipping occurs
    if total_norm > 10.0:
        print(f"Gradient clipped: {total_norm:.3f} → 10.0")

    optimizer.step()

    # update target network (soft or hard copy)
    update_target_network(q_network, target_q_network, tau=TAU)

    return loss.item()
    

In [276]:
# def get_action(q_values, valid_actions, epsilon=0.0):
#     """
#     Returns an action using an ε-greedy policy.

#     This function will return an action according to the following rules:
#         - With probability epsilon, it will return an action chosen at random.
#         - With probability (1 - epsilon), it will return the action that yields the
#         maximum Q value in q_values.
    
#     Args:
#         q_values (tf.Tensor):
#             The Q values returned by the Q-Network. For the Lunar Lander environment
#             this TensorFlow Tensor should have a shape of [1, 4] and its elements should
#             have dtype=tf.float32. 
#         epsilon (float):
#             The current value of epsilon.

#     Returns:
#        An action (numpy.int64). For the Lunar Lander environment, actions are
#        represented by integers in the closed interval [0,3].
#     """

#     if random.random() > epsilon:
#         # act = np.argmax(q_values.detach().numpy()[0])
#         act = torch.argmax(q_values, dim=1).item()
#         if act not in valid_actions:
#             return np.random.choice(valid_actions) # revise if there is any problem
#         else:
#             return act
#     else:
#         return np.random.choice(valid_actions) #random.choice(np.arange(4))

def get_action(q_values, valid_actions, epsilon=0.0):
    if random.random() > epsilon:
        mask = torch.full_like(q_values, -1e9)  # very negative
        mask[0, valid_actions] = q_values[0, valid_actions]
        act = torch.argmax(mask, dim=1).item()
        return act
    else:
        return np.random.choice(valid_actions)



In [277]:
def check_update_conditions(t, num_steps_upd, memory_buffer):
    """
    Determines if the conditions are met to perform a learning update.

    Checks if the current time step t is a multiple of num_steps_upd and if the
    memory_buffer has enough experience tuples to fill a mini-batch (for example, if the
    mini-batch size is 64, then the memory buffer should have more than 64 experience
    tuples in order to perform a learning update).
    
    Args:
        t (int):
            The current time step.
        num_steps_upd (int):
            The number of time steps used to determine how often to perform a learning
            update. A learning update is only performed every num_steps_upd time steps.
        memory_buffer (deque):
            A deque containing experiences. The experiences are stored in the memory
            buffer as namedtuples: namedtuple("Experience", field_names=["state",
            "action", "reward", "next_state", "done"]).

    Returns:
       A boolean that will be True if conditions are met and False otherwise. 
    """

    if (t + 1) % num_steps_upd == 0 and len(memory_buffer) > MINIBATCH_SIZE:
        return True
    else:
        return False


In [278]:
def get_new_eps(epsilon):
    """
    Updates the epsilon value for the ε-greedy policy.
    
    Gradually decreases the value of epsilon towards a minimum value (E_MIN) using the
    given ε-decay rate (E_DECAY).

    Args:
        epsilon (float):
            The current value of epsilon.

    Returns:
       A float with the updated value of epsilon.
    """

    return max(E_MIN, E_DECAY * epsilon)

In [279]:

def get_experiences(memory_buffer, device="cpu"):
    """
    Returns a random minibatch of experiences from memory_buffer
    as PyTorch tensors.

    Each state is already a vector (e.g. shape (100,)) that encodes
    information from the last timesteps.
    """

    experiences = random.sample(memory_buffer, k=MINIBATCH_SIZE)

    states = [e.state for e in experiences if e is not None]
    # torch.tensor(
    #     np.array([e.state for e in experiences if e is not None]),
    #     dtype=torch.float32,
    #     device=device
    # )
    actions = torch.tensor(
        np.array([e.action for e in experiences if e is not None]),
        dtype=torch.long,   # usually discrete, so long/int
        device=device
    )
    rewards = torch.tensor(
        np.array([e.reward for e in experiences if e is not None]),
        dtype=torch.float32,
        device=device
    )
    next_states = [e.next_state for e in experiences if e is not None]
    # torch.tensor(
    #     np.array([e.next_state for e in experiences if e is not None]),
    #     dtype=torch.float32,
    #     device=device
    # )
    dones = torch.tensor(
        np.array([e.done for e in experiences if e is not None]).astype(np.uint8),
        dtype=torch.float32,   # keep float for loss math
        device=device
    )

    return states, actions, rewards, next_states, dones


In [280]:
torch.cuda.is_available()

True

In [281]:
print(device)

cuda


In [282]:
memory_buffer = deque(maxlen=MEMORY_SIZE)
len(memory_buffer)
# memory_buffer.maxlen

0

In [283]:
lower_tf_data.shape[0]*.90

2700.0

In [284]:
higher_tf_data.tail()

,time,volume,mid_o,mid_h,mid_l,mid_c
495,2016-01-21 07:30:00+00:00,956,1.09156,1.09210,1.09022,1.09068
496,2016-01-21 08:00:00+00:00,1831,1.09070,1.09078,1.08854,1.08936
497,2016-01-21 08:30:00+00:00,1131,1.08940,1.09043,1.08928,1.08988
498,2016-01-21 09:00:00+00:00,1184,1.08986,1.09032,1.08953,1.08995
499,2016-01-21 09:30:00+00:00,1028,1.08998,1.09012,1.08889,1.08908


In [285]:
x_high[0][1]

tensor([ 3.0529, -0.2688,  0.2496, -0.2562, -0.0755,  0.2456,  0.6772, -2.2009,
        -1.3026,  0.5767,  1.4135,  0.5770, -1.8862, -0.3296], device='cuda:0')

In [286]:
env.window_generator.valid_idx(walk_forward_window=WALK_FORWARD_WINDOW)['ltf_range']

(26, 2508)

In [287]:
# env.window_generator.ltf_data.head(100)

In [288]:
# env.window_generator.cut_overlap()['htf_data']

In [289]:
env.window_generator.cut_overlap()['ltf_data']

,time,volume,mid_o,mid_h,mid_l,mid_c,force_index,elder_ray_bull,elder_ray_bear,stochastic_k,stochastic_d,rsi,ris,returns,returns_rolling_std,log_return,momentum,vol_ratio
0,2016-01-08 11:25:00+00:00,-0.245989,-0.356469,-0.370386,-0.334281,-0.297066,0.120373,0.370907,0.596589,1.596944,1.240865,0.505987,0.505987,0.523503,-0.215485,0.523660,0.609536,-0.777593
1,2016-01-08 11:30:00+00:00,-0.341243,-0.289853,-0.331205,-0.309972,-0.372830,0.042945,0.512981,0.665838,0.773616,1.294753,0.214985,0.214985,-0.582281,-0.192542,-0.582163,0.325821,-0.712548
2,2016-01-08 11:35:00+00:00,-0.547628,-0.380693,-0.424636,-0.316049,-0.384952,0.028956,0.000981,0.592347,0.687510,1.105535,0.168171,0.168171,-0.098571,-0.192304,-0.098375,0.172462,-0.710539
3,2016-01-08 11:40:00+00:00,-0.507939,-0.392804,-0.475873,-0.431516,-0.478899,-0.032497,-0.224439,0.018577,0.020187,0.535770,-0.189607,-0.189607,-0.720680,-0.298062,-0.720602,-0.463979,-0.856867
4,2016-01-08 11:45:00+00:00,-0.230113,-0.465476,-0.478887,-0.437593,-0.418288,0.025653,-0.254312,-0.028872,0.450718,0.419089,0.046808,0.046808,0.454541,-0.296551,0.454710,0.134122,-1.070903
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2555,2016-01-21 08:35:00+00:00,0.436670,0.430807,0.386105,0.388903,0.454508,-0.208437,-0.338631,-0.358049,-0.324722,-0.607171,-0.109606,-0.109606,0.246422,0.384964,0.246615,-0.962397,1.441890
2556,2016-01-21 08:40:00+00:00,0.555738,0.464115,0.470495,0.346363,0.375714,-0.318432,0.142200,-0.525771,-0.569980,-0.477643,-0.358478,-0.358478,-0.603953,0.400523,-0.603841,-0.486983,0.886715
2557,2016-01-21 08:45:00+00:00,0.579552,0.376303,0.497620,0.461830,0.436325,-0.165847,0.282099,0.086325,-0.381320,-0.460600,-0.141550,-0.141550,0.453348,0.132796,0.453517,-0.586667,0.837109
2558,2016-01-21 08:50:00+00:00,0.190595,0.445947,0.358980,0.346363,0.321165,-0.299998,-0.334127,-0.433479,-0.593425,-0.557697,-0.508836,-0.508836,-0.879790,0.163947,-0.879766,-0.394968,0.569505


In [290]:
# def train():
start = time.time()

num_episodes = 2000
# repeatation = 20
max_num_timesteps = WALK_FORWARD_WINDOW 

total_point_history = []
success_rate_history = []
pnl_history = []
balance_history = []
num_p_av = 20    # number of total points to use for averaging  # 100
epsilon = 1.0     # initial ε value for ε-greedy policy

# Create a memory buffer D with capacity N
memory_buffer = deque(maxlen=MEMORY_SIZE)

target_q_network.load_state_dict(q_network.state_dict())
#  TODO check if unnecessary device decleration 

step_counter = 0
valid_range = env.window_generator.valid_idx(walk_forward_window=WALK_FORWARD_WINDOW)['ltf_range']
for i in range(num_episodes):
    
    # Reset the environment to the initial state and get the initial state
    # try:
    #     state = env.reset()
    # except:
    #     env.window_generator.reset()
    #     state = env.reset()
    # x_low, x_high, pos_info = state_to_tensor(current_state)
    
    # n_candle_to_train

    start_index = np.random.randint(valid_range[0], valid_range[1]) # 15 is the k period..
    # print('\t',start_index)
    # of stochastic that makes nan in ltfdata
    # print("start_index", start_index)
    env.window_generator.reset(idx_low=start_index)
    state = env.reset()
    

    total_points = 0
    
    for t in range(max_num_timesteps):
        # print('t', t)
        # print(' higher last index',env.window_generator.higher_last_candle_index)
        # print(' lower last index',env.window_generator.lower_last_candle_index)
        # if step_counter >= lower_tf_data.shape[0] * 0.90:
        #     env.window_generator.reset()
        #     print(f'Window generator reset after {step_counter} steps.')
        #     step_counter = 0
        #     break

        # From the current state S choose an action A using an ε-greedy policy
        x_low, x_high, pos_info = state_to_tensor(state, device=device)    
        # print('\nx_low shape', x_low.shape)
        # print('x_high shape', x_high.shape)    
        # print('pos shape', pos_info.shape)
        # print(state_qn)
        q_values = q_network(x_low, x_high, pos_info)

        valid_actions = env._get_valid_actions()
        action = get_action(q_values, valid_actions, epsilon)
        # action = np.random.choice(valid_actions)
        
        
        next_state, reward, done, info = env.step(action)
        # print('current_price',env.get_current_price())
        # print(next_state['htf_data'], '\n')
        # print(next_state['position_info'])
        # try:
        #     next_state, reward, done, info = env.step(action)
        # except:
        #     # env.window_generator.reset()
        #     # next_state, reward, done, info = env.step(action)
        #     break
        
        # print(f" action: {env.action_to_name(action)}, reward: {reward}")
        # if done:
        #     break
        # print()
        
        # Store experience tuple (S,A,R,S') in the memory buffer.
        # We store the done variable as well for convenience.
        # if step_counter <= lower_tf_data.shape[0] * 0.98:
        # Store packed numeric arrays in replay to avoid huge DataFrame objects.
        memory_buffer.append(
            experience(pack_state(state), action, reward, pack_state(next_state), done)
        )
        step_counter += 1 
        
        # Only update the network every NUM_STEPS_FOR_UPDATE time steps.
        update = check_update_conditions(t, NUM_STEPS_FOR_UPDATE, memory_buffer)
        
        if update:
            # Sample random mini-batch of experience tuples (S,A,R,S') from D
            experiences = get_experiences(memory_buffer, device=device)
            
            # Set the y targets, perform a gradient descent step,
            # and update the network weights.
            agent_learn(experiences, GAMMA, device=device)
        # print(reward)

        state = next_state.copy()
        total_points += reward
        
        if done:
            # print(f'\tdone in step {t}, reward:{reward}')
            break
            
        # if t> 10: 
        #     break

    # if i % num_episodes == 0:
    #     env.window_generator.alignment(0)

    success_counter = 0
    total_pnl_per_episode = 0

    for trade in env.trade_history:
        if trade['success']:
            success_counter += 1
        total_pnl_per_episode += trade['pnl']  
    
    mean_pnl = total_pnl_per_episode/(len(env.trade_history)+1)
    pnl_history.append(mean_pnl)

    success_rate = success_counter/(len(env.trade_history)+1)
    success_rate_history.append(success_rate)

    total_point_history.append(total_points)
    av_latest_points = np.mean(total_point_history[-num_p_av:])

    if av_latest_points > 10000:
        break
    
    epsilon = get_new_eps(epsilon)
    print(f"\rEpisode {i+1} | Total point average of the last {num_p_av} episodes: {av_latest_points:.5f}", end="")
    # eplilon: {epsilon:.5f}, memory: {len(memory_buffer)}
    if (i+1) % num_p_av == 0:
        print(f"\rEpisode {i+1} | Total point average of the last {num_p_av} episodes: {av_latest_points:.5f}")

    # We will consider that the environment is solved if we get an
    # average of 200 points in the last 100 episodes.
    # if av_latest_points >= 200.0:
    #     print(f"\n\nEnvironment solved in {i+1} episodes!")
    #     q_network.save('lunar_lander_model.h5')
    #     break
        
         
tot_time = time.time() - start



print(f"\nTotal Runtime: {tot_time:.5f} s ({(tot_time/60):.5f} min)")
    # print(q_values, action)
torch.save(q_network.state_dict(), "test_env_data/converged_model.pth")
torch.save(q_network.state_dict(), "converged_model.pth")

# train()


Episode 1 | Total point average of the last 20 episodes: 0.00058

Episode 20 | Total point average of the last 20 episodes: -0.00001
Episode 40 | Total point average of the last 20 episodes: -0.00004
Episode 60 | Total point average of the last 20 episodes: -0.00013
Episode 80 | Total point average of the last 20 episodes: -0.00012
Episode 100 | Total point average of the last 20 episodes: -0.00024
Episode 120 | Total point average of the last 20 episodes: -0.00014
Episode 140 | Total point average of the last 20 episodes: -0.00009
Episode 160 | Total point average of the last 20 episodes: -0.00015
Episode 180 | Total point average of the last 20 episodes: -0.00022
Episode 200 | Total point average of the last 20 episodes: -0.00006
Episode 220 | Total point average of the last 20 episodes: -0.00006
Episode 240 | Total point average of the last 20 episodes: -0.00033
Episode 260 | Total point average of the last 20 episodes: -0.00019
Episode 280 | Total point average of the last 20 episodes: -0.00012
Episode 300 | Total point average of the last 20 epi

KeyboardInterrupt: 

In [ ]:
valid_range

In [ ]:
env.lower_tf_data

In [ ]:
env.higher_tf_data

In [ ]:
one_counter = 0
two_counter = 0
zero_counter = 0
three_counter = 0
for e in memory_buffer:
    if e.action == 1:
        one_counter +=1
    elif e.action == 2:
        two_counter +=1
    elif e.action == 0:
        zero_counter +=1
    else:
        three_counter +=1

print('ones', one_counter)
print('twos', two_counter)
print('zeros', zero_counter)
print('threes', three_counter)

In [ ]:
# total_point_history

In [ ]:
def plot_history(point_history, y_label,**kwargs):
    """
    Plots the total number of points received by the agent after each episode together
    with the moving average (rolling mean). 

    Args:
        point_history (list):
            A list containing the total number of points the agent received after each
            episode.
        **kwargs: optional
            window_size (int):
                Size of the window used to calculate the moving average (rolling mean).
                This integer determines the fixed number of data points used for each
                window. The default window size is set to 10% of the total number of
                data points in point_history, i.e. if point_history has 200 data points
                the default window size will be 20.
            lower_limit (int):
                The lower limit of the x-axis in data coordinates. Default value is 0.
            upper_limit (int):
                The upper limit of the x-axis in data coordinates. Default value is
                len(point_history).
            plot_rolling_mean_only (bool):
                If True, only plots the moving average (rolling mean) without the point
                history. Default value is False.
            plot_data_only (bool):
                If True, only plots the point history without the moving average.
                Default value is False.
    """

    lower_limit = 0
    upper_limit = len(point_history)

    window_size = (upper_limit * 10) // 100

    plot_rolling_mean_only = False
    plot_data_only = False

    if kwargs:
        if "window_size" in kwargs:
            window_size = kwargs["window_size"]

        if "lower_limit" in kwargs:
            lower_limit = kwargs["lower_limit"]

        if "upper_limit" in kwargs:
            upper_limit = kwargs["upper_limit"]

        if "plot_rolling_mean_only" in kwargs:
            plot_rolling_mean_only = kwargs["plot_rolling_mean_only"]

        if "plot_data_only" in kwargs:
            plot_data_only = kwargs["plot_data_only"]

    points = point_history[lower_limit:upper_limit]

    # Generate x-axis for plotting.
    episode_num = [x for x in range(lower_limit, upper_limit)]

    # Use Pandas to calculate the rolling mean (moving average).
    rolling_mean = pd.DataFrame(points).rolling(window_size).mean()

    plt.figure(figsize=(10, 7), facecolor="white")

    if plot_data_only:
        plt.plot(episode_num, points, linewidth=1, color="cyan")
    elif plot_rolling_mean_only:
        plt.plot(episode_num, rolling_mean, linewidth=2, color="magenta")
    else:
        plt.plot(episode_num, points, linewidth=1, color="cyan")
        plt.plot(episode_num, rolling_mean, linewidth=2, color="magenta")

    text_color = "black"

    ax = plt.gca()
    ax.set_facecolor("black")
    plt.grid()
    plt.xlabel("Episode", color=text_color, fontsize=30)
    plt.ylabel(y_label, color=text_color, fontsize=30)
    yNumFmt = mticker.StrMethodFormatter("{x:,}")
    ax.yaxis.set_major_formatter(yNumFmt)
    ax.tick_params(axis="x", colors=text_color)
    ax.tick_params(axis="y", colors=text_color)
    plt.show()

In [ ]:
try:
    plot_history(total_point_history, "Total Points")
    plot_history(pnl_history, "Mean Pnalty & Loss")
    plot_history(success_rate_history, "Success Rate")
except:
    print('No history to plot!')

# Live Testing

## Testing the Model

In [ ]:
lower_tf_data = m5_data_filtered[:12000].copy()
higher_tf_data = m30_data_filtered[:2000].copy()
n_candles_to_test = 200

env = MultiTimeframeForexEnv(
    lower_tf_data=lower_tf_data,
    higher_tf_data=higher_tf_data,
    walk_forward_window=n_candles_to_test,
    lookback_window=LOOKBACK_WINDOW_SIZE,
    # walk_forward_window=100,
    stop_loss_pct=STOP_LOSS_PCT, #0.0015,
    take_profit_pct=TAKE_PROFIT_PCT,
    test=True,
    n_candles_to_test=n_candles_to_test
)




current_state = env.reset()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

x_low, x_high, pos_info = state_to_tensor(current_state, device=device)

q_network = QModel(
    input_dim_low=x_low.shape[2], 
    input_dim_high=x_high.shape[2],
    pos_info_dim=pos_info.shape[1],
    fusion_dim=64
).to(device)

state_dict = torch.load("converged_model.pth", map_location=device)
q_network.load_state_dict(state_dict)

q_network.eval()

In [ ]:
# current_state['ltf_data']

In [ ]:
action_list = []
current_state = env.reset()
with torch.no_grad():
    for t in range(n_candles_to_test-2):
        # print(current_state['ltf_data'])
        
        # From the current state S choose an action A using an ε-greedy policy
        x_low, x_high, pos_info = state_to_tensor(current_state, device=device)
        q_values = q_network(x_low, x_high, pos_info)
        print('\nq values',q_values[0])
        valid_actions = env._get_valid_actions()
        print('current position',env.current_position)
        print('valid actions', valid_actions)
        
        action = get_action(q_values, valid_actions, epsilon=0)
        print('chosen action', action)
        action_list.append(action)
        # action = np.random.choice(valid_actions)
        
        next_state, _, _, _ = env.step(action)
        current_state = next_state.copy()


In [ ]:
env.current_balance

In [ ]:
# len(action_list)
action_list[-10:]

In [ ]:
q_vals = q_network(x_low, x_high, pos_info)
print('q_vals',q_vals)
mask = torch.full_like(q_values, -1e9)  # very negative
print('mask',mask)
print('valid_actions', valid_actions)

mask[0, valid_actions] = q_values[0, valid_actions]
print('after masking', mask)
act = torch.argmax(mask, dim=1).item()
print(act)

In [ ]:
import pandas as pd
import mplfinance as mpf
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np


data = env.window_generator.get_test_df()
df = pd.DataFrame(data).set_index("time")
df['Open'] = df['mid_o']
df['High'] = df['mid_h']
df['Close'] = df['mid_c']
df['Low'] = df['mid_l']

# --- Example agent actions (0: noop, 1: buy, 2: sell, 3: hold) ---
actions = action_list #np.random.choice([0,1,2,3], size=len(df))

# --- Parameters ---
update_interval = 1  # seconds per candle

# --- Matplotlib setup ---
fig, ax = plt.subplots(figsize=(10,5))
ax.set_title("Live Candle Simulation")

def update(n):
    
    ax.clear()
    ax.set_title("Live Candle Simulation")

    # take first N candles (ensure n>=1)
    start = 0
    if n > 100:
        start = n-100
    df_partial = df.iloc[start:n]
    act_partial = actions[start:n]

    # plot candlesticks
    mpf.plot(df_partial, type='candle', ax=ax, volume=False, style='charles', show_nontrading=True)

    # overlay agent actions on CLOSE price
    for date, close, high, low, act in zip(df_partial.index, 
                                           df_partial['Close'], 
                                           df_partial['High'], 
                                           df_partial['Low'], 
                                           act_partial):
        if act == 1:   # Buy → show above high
            ax.scatter(date, high * 1.002, marker="^", color="green", s=80, label="Buy" if n==1 else "")
        elif act == 2: # Sell → show below low
            ax.scatter(date, low * 0.998, marker="v", color="red", s=80, label="Sell" if n==1 else "")
        elif act == 3: # Hold → show just above close
            ax.scatter(date, close * 1.001, marker="o", color="blue", s=60, label="Hold" if n==1 else "")
        # 0 = noop → nothing

    if n == 1:  # only show legend once
        ax.legend()

ani = FuncAnimation(fig, update,
                    frames=range(1, len(df)+1),  # <-- start from 1
                    interval=update_interval*1000,
                    repeat=True)

HTML(ani.to_jshtml())